# Visual Product Search — Condition B (Frozen CLIP + BLIP-2)

This notebook implements **Condition B** from the ablation study:

- CLIP: frozen (no fine-tuning)
- BLIP-2: frozen (used for caption generation)
- Fusion: image + text embeddings using α-weighting

We evaluate retrieval performance using:
- Recall@K
- NDCG@K
- mAP@K

Results are averaged across multiple seeds (roll numbers).

## Setup

In [ ]:
!pip install git+https://github.com/openai/CLIP.git -q
!pip install transformers accelerate -q
!pip install faiss-cpu -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 79.4 MB/s eta 0:00:00


In [2]:
import os, json, math, random, zipfile
import numpy as np
import pandas as pd
import torch
import clip
import faiss
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import Blip2Processor, Blip2ForConditionalGeneration

SEEDS = [2023006, 2023008, 2023524, 2023552]
ALPHAS = [0.7, 0.85]

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Dataset Loading

We use the DeepFashion In-Shop dataset.
- Query split → input images
- Gallery split → retrieval database

In [3]:
ROOT     = '/kaggle/input/datasets/dedeepyaavancha/deepfashion-in-shop-clothes-retrieval-benchmark/In-shop Clothes Retrieval Benchmark'
SAVE_DIR = '/kaggle/working/condition_b'
os.makedirs(SAVE_DIR, exist_ok=True)
print('ROOT exists:', os.path.exists(ROOT))

eval_path = os.path.join(ROOT, 'Eval/list_eval_partition.txt')
bbox_path = os.path.join(ROOT, 'Anno/list_bbox_inshop.txt')

df = pd.read_csv(eval_path, sep=r'\s+', skiprows=2, header=None,
                 names=['image_name','item_id','split'])

df_bbox = pd.read_csv(bbox_path, sep=r'\s+', skiprows=2, header=None,
                      names=['image_name','clothes_type','pose_type','x1','y1','x2','y2'])

df = df.merge(df_bbox, on='image_name')
df['image_path'] = df['image_name'].apply(lambda x: os.path.join(ROOT,x))

df_query = df[df['split']=='query'].reset_index(drop=True)
df_gallery = df[df['split']=='gallery'].reset_index(drop=True)

gallery_ids_arr = np.array(df_gallery['item_id'].tolist())
query_ids = df_query['item_id'].tolist()

ROOT exists: True


## Models

In [4]:
clip_model, preprocess = clip.load('ViT-B/32', device=device)
clip_model.eval()

blip_processor = Blip2Processor.from_pretrained('Salesforce/blip2-opt-2.7b')
blip_model = Blip2ForConditionalGeneration.from_pretrained(
    'Salesforce/blip2-opt-2.7b',
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
).to(device)
blip_model.eval()

100%|███████████████████████████████████████| 338M/338M [00:03<00:00, 98.0MiB/s]


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Blip2ForConditionalGeneration(
  (vision_model): Blip2VisionModel(
    (embeddings): Blip2VisionEmbeddings(
      (patch_embedding): Conv2d(3, 1408, kernel_size=(14, 14), stride=(14, 14))
    )
    (encoder): Blip2Encoder(
      (layers): ModuleList(
        (0-38): 39 x Blip2EncoderLayer(
          (self_attn): Blip2Attention(
            (qkv): Linear(in_features=1408, out_features=4224, bias=True)
            (projection): Linear(in_features=1408, out_features=1408, bias=True)
          )
          (layer_norm1): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
          (mlp): Blip2MLP(
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=1408, out_features=6144, bias=True)
            (fc2): Linear(in_features=6144, out_features=1408, bias=True)
          )
          (layer_norm2): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
        )
      )
    )
    (post_layernorm): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
  )
  (qf

## Helper Functions

In [5]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


class FashionDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        return preprocess(img), row['item_id']


def embed_split(df):
    loader = DataLoader(
        FashionDataset(df),
        batch_size=256,
        pin_memory=True,
        num_workers=2
    )

    embs = []
    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(device, non_blocking=True)
            e = clip_model.encode_image(imgs)
            e = e / e.norm(dim=-1, keepdim=True)
            embs.append(e.cpu().numpy())

    torch.cuda.empty_cache()
    return np.vstack(embs)



def generate_captions(df, batch_size=16):
    caps = []

    for i in range(0, len(df), batch_size):
        batch = df.iloc[i:i+batch_size]

        images = [
            Image.open(row['image_path']).convert('RGB')
            for _, row in batch.iterrows()
        ]
        inputs = blip_processor(
            images=images,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            out = blip_model.generate(**inputs, max_new_tokens=40)

        decoded = blip_processor.batch_decode(out, skip_special_tokens=True)
        caps.extend(decoded)

        torch.cuda.empty_cache()

    return caps


def encode_text(caps, batch_size=256):
    all_embs = []

    for i in range(0, len(caps), batch_size):
        batch_caps = caps[i:i+batch_size]

        tokens = clip.tokenize(batch_caps, truncate=True).to(device)

        with torch.no_grad():
            e = clip_model.encode_text(tokens)
            e = e / e.norm(dim=-1, keepdim=True)

        all_embs.append(e.cpu().numpy())

        torch.cuda.empty_cache()

    return np.vstack(all_embs)


def fuse(img, txt, alpha):
    f = alpha * img + (1 - alpha) * txt
    return f / np.linalg.norm(f, axis=1, keepdims=True)

## Metrics

In [6]:
def compute_metrics(indices, Ks=[5,10,15]):
    results = {}
    for K in Ks:
        recalls, ndcgs, aps = [], [], []

        for i, qid in enumerate(query_ids):
            retrieved = gallery_ids_arr[indices[i,:K]]
            relevant = (retrieved == qid)

            recalls.append(1.0 if relevant.any() else 0.0)

            dcg = sum(rel / math.log2(idx+2) for idx, rel in enumerate(relevant))
            idcg = sum(1.0 / math.log2(idx+2) for idx in range(min(int(relevant.sum()), K)))
            ndcgs.append(dcg/idcg if idcg>0 else 0)

            hits, prec = 0, 0
            for idx, rel in enumerate(relevant):
                if rel:
                    hits += 1
                    prec += hits/(idx+1)
            aps.append(prec / max(1, int(relevant.sum())))

        results[K] = {
            "Recall": np.mean(recalls),
            "NDCG": np.mean(ndcgs),
            "mAP": np.mean(aps)
        }
    return results

def run_retrieval(g, q, label):
    index = faiss.IndexFlatIP(g.shape[1])
    index.add(g.astype(np.float32))

    sims, idxs = index.search(q.astype(np.float32), 15)
    metrics = compute_metrics(idxs)

    print(f"\n===== {label} =====")
    print(f"{'K':<5} {'Recall':<10} {'NDCG':<10} {'mAP':<10}")
    print("-"*40)
    for K, v in metrics.items():
        print(f"{K:<5} {v['Recall']:.4f}   {v['NDCG']:.4f}   {v['mAP']:.4f}")

    return metrics

## Multi-Seed Evaluation

We run experiments across multiple seeds (roll numbers) and report mean ± std.

In [7]:
all_results = []

# ---- (OPTIONAL BUT IMPORTANT) GLOBAL DETERMINISM ----
def set_global_determinism(seed):
    import random, numpy as np, torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ---- COMPUTE IMAGE EMBEDDINGS ONCE (NO REDUNDANCY) ----
set_global_determinism(SEEDS[0])

g_img = embed_split(df_gallery)
q_img = embed_split(df_query)

np.save(f"{SAVE_DIR}/gallery_img.npy", g_img)
np.save(f"{SAVE_DIR}/query_img.npy", q_img)


# ---- MAIN LOOP ----
for seed in SEEDS:
    print(f"\n========== SEED {seed} ==========")
    set_global_determinism(seed)

    # ---- CAPTION GENERATION (MAKE SURE THIS IS DETERMINISTIC) ----
    g_cap = generate_captions(df_gallery)
    q_cap = generate_captions(df_query)

    with open(f"{SAVE_DIR}/gallery_caps_seed{seed}.json", "w") as f:
        json.dump(g_cap, f)
    with open(f"{SAVE_DIR}/query_caps_seed{seed}.json", "w") as f:
        json.dump(q_cap, f)

    # ---- TEXT ENCODING ----
    g_txt = encode_text(g_cap)
    q_txt = encode_text(q_cap)

    np.save(f"{SAVE_DIR}/gallery_txt_seed{seed}.npy", g_txt)
    np.save(f"{SAVE_DIR}/query_txt_seed{seed}.npy", q_txt)

    seed_res = {"seed": seed, "alphas": {}}

    for alpha in ALPHAS:
        alpha_key = round(alpha, 3)

        g_f = fuse(g_img, g_txt, alpha)
        q_f = fuse(q_img, q_txt, alpha)

        np.save(f"{SAVE_DIR}/gallery_fused_seed{seed}_alpha{alpha_key}.npy", g_f)
        np.save(f"{SAVE_DIR}/query_fused_seed{seed}_alpha{alpha_key}.npy", q_f)

        m = run_retrieval(g_f, q_f, f"Seed {seed} | Alpha {alpha_key}")
        seed_res["alphas"][alpha_key] = m

    all_results.append(seed_res)


# ---- FLATTEN RESULTS ----
flat_results = []

for res in all_results:
    row = {"seed": res["seed"]}

    for alpha_key, metrics in res["alphas"].items():
        for K in [5, 10, 15]:
            row[f"alpha_{alpha_key}_R@{K}"] = metrics[K]["Recall"]
            row[f"alpha_{alpha_key}_NDCG@{K}"] = metrics[K]["NDCG"]
            row[f"alpha_{alpha_key}_mAP@{K}"] = metrics[K]["mAP"]

    flat_results.append(row)


# ---- DATAFRAME + STATS ----
df_res = pd.DataFrame(flat_results)

mean_res = df_res.drop(columns=["seed"]).mean()
std_res  = df_res.drop(columns=["seed"]).std()

print("\n===== FINAL MEAN ± STD =====")
for col in mean_res.index:
    print(f"{col}: {mean_res[col]:.4f} ± {std_res[col]:.4f}")


# ---- SAVE RESULTS ----
df_res.to_csv(f"{SAVE_DIR}/results_per_seed.csv", index=False)

summary_df = pd.DataFrame({
    "metric": mean_res.index,
    "mean": mean_res.values,
    "std": std_res.values
})

summary_df.to_csv(f"{SAVE_DIR}/results_summary.csv", index=False)


========== SEED 2023006 ==========

===== Seed 2023006 | Alpha 0.7 =====
K     Recall     NDCG       mAP       
----------------------------------------
5     0.6281   0.5196   0.4752
10    0.7144   0.5359   0.4594
15    0.7595   0.5405   0.4452

===== Seed 2023006 | Alpha 0.85 =====
K     Recall     NDCG       mAP       
----------------------------------------
5     0.6869   0.5730   0.5252
10    0.7638   0.5860   0.5063
15    0.8021   0.5872   0.4885

========== SEED 2023008 ==========

===== Seed 2023008 | Alpha 0.7 =====
K     Recall     NDCG       mAP       
----------------------------------------
5     0.6281   0.5196   0.4752
10    0.7144   0.5359   0.4594
15    0.7595   0.5405   0.4452

===== Seed 2023008 | Alpha 0.85 =====
K     Recall     NDCG       mAP       
----------------------------------------
5     0.6869   0.5730   0.5252
10    0.7638   0.5860   0.5063
15    0.8021   0.5872   0.4885

========== SEED 2023524 ==========

===== Seed 2023524 | Alpha 0.7 =====
K     Re

In [8]:
zip_path = '/kaggle/working/condition_b_outputs.zip'

with zipfile.ZipFile(zip_path, 'w') as z:
    for fname in os.listdir(SAVE_DIR):
        z.write(os.path.join(SAVE_DIR, fname), fname)

print("Saved zip:", zip_path)

Saved zip: /kaggle/working/condition_b_outputs.zip
